In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import from_json, col, expr
from pyspark.sql.types import *
from config import KAFKA_BROKER, TOPICS

spark = SparkSession.builder \
    .appName("FlightDelayStream") \
    .config("spark.jars.packages",
            "org.apache.spark:spark-sql-kafka-0-10_2.12:3.4.0") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

# Schema الرحلات
flight_schema = StructType([
    StructField("icao24", StringType()),
    StructField("callsign", StringType()),
    StructField("origin_country", StringType()),
    StructField("longitude", DoubleType()),
    StructField("latitude", DoubleType()),
    StructField("altitude", DoubleType()),
    StructField("velocity", DoubleType()),
    StructField("heading", DoubleType()),
    StructField("on_ground", BooleanType()),
    StructField("timestamp", LongType())
])

# Schema الطقس
weather_schema = StructType([
    StructField("airport", StringType()),
    StructField("latitude", DoubleType()),
    StructField("longitude", DoubleType()),
    StructField("temperature", DoubleType()),
    StructField("wind_speed", DoubleType()),
    StructField("precipitation", DoubleType()),
    StructField("weathercode", IntegerType()),
    StructField("timestamp", StringType())
])

# اقرأ flights من Kafka
flights_df = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", KAFKA_BROKER) \
    .option("subscribe", TOPICS["flights"]) \
    .option("startingOffsets", "latest") \
    .load() \
    .select(from_json(col("value").cast("string"), flight_schema).alias("data")) \
    .select("data.*")

# اقرأ weather من Kafka
weather_df = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", KAFKA_BROKER) \
    .option("subscribe", TOPICS["weather"]) \
    .option("startingOffsets", "latest") \
    .load() \
    .select(from_json(col("value").cast("string"), weather_schema).alias("data")) \
    .select("data.*")

# اطبع flights
flights_query = flights_df.writeStream \
    .outputMode("append") \
    .format("console") \
    .option("truncate", False) \
    .queryName("flights_stream") \
    .start()

# اطبع weather
weather_query = weather_df.writeStream \
    .outputMode("append") \
    .format("console") \
    .option("truncate", False) \
    .queryName("weather_stream") \
    .start()

print("Consumer started - waiting for data...")
spark.streams.awaitAnyTermination()